In [1]:
# Import PuLP
from pulp import LpProblem, LpVariable, LpMinimize, lpSum, value

# Step 1: Define the Problem
# Create the Linear Programming problem
model = LpProblem("Minimize_Shipping_Costs", LpMinimize)

# Step 2: Input Data
# Define costs as a dictionary
costs = {
    ('Seattle', 'LA'): 100,
    ('Seattle', 'Chicago'): 120,
    ('Dallas', 'Chicago'): 110,
    ('Dallas', 'Atlanta'): 90,
    ('Miami', 'NY'): 95,
    ('Miami', 'Atlanta'): 85
}

# Define warehouse capacities explicitly
capacities = {
    'Seattle': 400,
    'Dallas': 300,
    'Miami': 200
}

# Define supermarket demands explicitly
demands = {
    'LA': 200,
    'Chicago': 150,
    'Atlanta': 250,
    'NY': 100
}

# Step 3: Define Decision Variables
# Explicitly declare the valid routes
routes = [
    ('Seattle', 'LA'),
    ('Seattle', 'Chicago'),
    ('Dallas', 'Chicago'),
    ('Dallas', 'Atlanta'),
    ('Miami', 'NY'),
    ('Miami', 'Atlanta')
]

# Create decision variables for each route
vars = LpVariable.dicts("Route", routes, lowBound=0, cat='Continuous')

# Step 4: Define the Objective Function
# Minimize total shipping cost
model += lpSum(vars[(w, s)] * costs[(w, s)] for (w, s) in routes), "Total_Shipping_Cost"

# Step 5: Add Constraints
# Warehouse capacity constraints
model += vars[('Seattle', 'LA')] + vars[('Seattle', 'Chicago')] <= capacities['Seattle'], "Seattle_Capacity"
model += vars[('Dallas', 'Chicago')] + vars[('Dallas', 'Atlanta')] <= capacities['Dallas'], "Dallas_Capacity"
model += vars[('Miami', 'NY')] + vars[('Miami', 'Atlanta')] <= capacities['Miami'], "Miami_Capacity"

# Supermarket demand constraints
model += vars[('Seattle', 'LA')] + vars[('Dallas', 'Chicago')] + vars[('Miami', 'Atlanta')] == demands['LA'], "LA_Demand"
model += vars[('Seattle', 'Chicago')] + vars[('Dallas', 'Chicago')] == demands['Chicago'], "Chicago_Demand"
model += vars[('Dallas', 'Atlanta')] + vars[('Miami', 'Atlanta')] == demands['Atlanta'], "Atlanta_Demand"
model += vars[('Miami', 'NY')] == demands['NY'], "NY_Demand"

# Step 6: Solve the Model
model.solve()

# Step 7: Print Results
print("Optimal Total Cost:", value(model.objective))
print("Optimal Shipping Quantities:")
for (w, s) in routes:
    print(f"{w} to {s}: {vars[(w, s)].varValue}")


Optimal Total Cost: 48375.0
Optimal Shipping Quantities:
Seattle to LA: 0.0
Seattle to Chicago: 25.0
Dallas to Chicago: 125.0
Dallas to Atlanta: 175.0
Miami to NY: 100.0
Miami to Atlanta: 75.0
